# Validación y métrica del negocio: emergencias por cantón

El modelo final de la sesión pasada (gradient boosting con `max_depth=5`) y el segundo (la red neuronal), evaluados con validación cruzada y dos esquemas:

- **Temporal**, el que corresponde: el modelo predice el futuro (`obj_n_emergencias_t7`, las emergencias dentro de 7 días). Cada pliegue entrena con las fechas anteriores y valida con las siguientes. Los pliegues se cortan por fecha, no por fila, porque cada fecha tiene una fila por cantón, y entre entrenamiento y validación quedan 7 días de separación.
- **Al azar**, para comparar: mezcla días de todo el período en entrenamiento y validación.

La preparación y las features se importan de `pipeline_proyecto.py` (Semana 9, Sesión 2).

In [1]:
import sys
import time

import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

sys.path.insert(0, "../../semana-09-sesion-02/plantilla-pipeline")
from pipeline_proyecto import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, cargar_datos, construir_preparacion

datos = cargar_datos().sort_values(["fecha", "cod_canton"]).reset_index(drop=True)
X = datos[NUMERICAS + CATEGORICAS + YA_LISTAS]
y = datos[OBJETIVO].to_numpy()
print("Filas:", len(datos), "| fechas:", datos["fecha"].nunique(), "| desde", datos["fecha"].min().date(), "hasta", datos["fecha"].max().date())

Filas: 404484 | fechas: 1822 | desde 2021-07-29 hasta 2026-08-24


## La métrica del negocio

Subestimar es el error caro: si el modelo predice menos emergencias de las que ocurren, el cantón queda con menos personal y unidades de las necesarias, y una emergencia sin atender a tiempo cuesta mucho más que una unidad que queda ociosa. Supongo, de forma aproximada, que **cada emergencia subestimada cuesta 3 veces lo que una sobreestimada**.

- `costo_asimetrico`: promedio por cantón y día de `3 × emergencias subestimadas + 1 × emergencias sobreestimadas`.
- `subestimados_20`: proporción de cantón-días en que las emergencias reales superan a las predichas en más de 20 %.
- `mae`: la métrica de siempre, que pesa igual los dos errores.

In [2]:
COSTO_SUBESTIMAR = 3
COSTO_SOBREESTIMAR = 1


def metricas(real, predicho):
    faltan = np.clip(real - predicho, 0, None)
    sobran = np.clip(predicho - real, 0, None)
    return {
        "mae": np.mean(np.abs(real - predicho)),
        "costo_asimetrico": np.mean(COSTO_SUBESTIMAR * faltan + COSTO_SOBREESTIMAR * sobran),
        "subestimados_20": np.mean(real > predicho * 1.2),
    }

## Los dos esquemas

In [3]:
# Temporal: TimeSeriesSplit sobre las fechas (no sobre las filas), con 7 dias
# de separacion porque el objetivo de cada fila es t+7.
fechas = np.sort(datos["fecha"].unique())


def pliegues_temporales(n_pliegues=5, separacion_dias=7):
    for indices_ent, indices_val in TimeSeriesSplit(n_splits=n_pliegues).split(fechas):
        inicio_val = fechas[indices_val[0]]
        fechas_ent = fechas[indices_ent]
        fechas_ent = fechas_ent[fechas_ent < inicio_val - np.timedelta64(separacion_dias, "D")]
        ent = np.flatnonzero(datos["fecha"].isin(fechas_ent))
        val = np.flatnonzero(datos["fecha"].isin(fechas[indices_val]))
        yield ent, val


esquemas = {
    "temporal": list(pliegues_temporales()),
    "al azar": list(KFold(n_splits=5, shuffle=True, random_state=42).split(X)),
}
for numero, (ent, val) in enumerate(esquemas["temporal"]):
    print(f"Temporal, pliegue {numero + 1}: entrena hasta {datos['fecha'].iloc[ent].max().date()} ({len(ent)} filas), "
          f"valida del {datos['fecha'].iloc[val].min().date()} al {datos['fecha'].iloc[val].max().date()} ({len(val)} filas)")

Temporal, pliegue 1: entrena hasta 2022-05-24 (66600 filas), valida del 2022-06-01 al 2023-03-30 (67266 filas)
Temporal, pliegue 2: entrena hasta 2023-03-23 (133866 filas), valida del 2023-03-31 al 2024-02-27 (67266 filas)
Temporal, pliegue 3: entrena hasta 2024-02-20 (201132 filas), valida del 2024-02-28 al 2024-12-26 (67266 filas)
Temporal, pliegue 4: entrena hasta 2024-12-19 (268398 filas), valida del 2024-12-27 al 2025-10-25 (67266 filas)
Temporal, pliegue 5: entrena hasta 2025-10-18 (335664 filas), valida del 2025-10-26 al 2026-08-24 (67266 filas)


## Validación cruzada de los dos modelos (y la línea base como referencia)

In [4]:
modelos = {
    "gradient boosting max_depth=5": lambda: GradientBoostingRegressor(max_depth=5, random_state=42),
    "red neuronal (64)": lambda: MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=42),
}

filas = []
for esquema, pliegues in esquemas.items():
    for numero, (ent, val) in enumerate(pliegues):
        # Linea base "igual que hoy": no entrena, sirve de referencia.
        filas.append({"esquema": esquema, "modelo": "linea base (igual que hoy)", "pliegue": numero + 1,
                      **metricas(y[val], datos["n_emergencias"].to_numpy()[val])})
        for nombre, crear in modelos.items():
            pipeline = Pipeline([
                ("preparar", construir_preparacion()),
                ("modelo", TransformedTargetRegressor(regressor=crear(), func=np.log1p, inverse_func=np.expm1)),
            ])
            inicio = time.perf_counter()
            pipeline.fit(X.iloc[ent], y[ent])
            filas.append({"esquema": esquema, "modelo": nombre, "pliegue": numero + 1,
                          **metricas(y[val], pipeline.predict(X.iloc[val])),
                          "segundos": time.perf_counter() - inicio})
        print(f"{esquema}, pliegue {numero + 1} listo")

por_pliegue = pd.DataFrame(filas)

temporal, pliegue 1 listo


temporal, pliegue 2 listo


temporal, pliegue 3 listo


temporal, pliegue 4 listo


temporal, pliegue 5 listo


al azar, pliegue 1 listo


al azar, pliegue 2 listo


al azar, pliegue 3 listo


al azar, pliegue 4 listo


al azar, pliegue 5 listo


## La tabla: promedio, desviación y cada pliegue

In [5]:
def resumir(grupo):
    return pd.Series({
        "mae_promedio": grupo["mae"].mean(),
        "mae_desviacion": grupo["mae"].std(ddof=0),
        "mae_pliegues": [round(valor, 2) for valor in grupo["mae"]],
        "costo_promedio": grupo["costo_asimetrico"].mean(),
        "costo_desviacion": grupo["costo_asimetrico"].std(ddof=0),
        "costo_pliegues": [round(valor, 2) for valor in grupo["costo_asimetrico"]],
        "subestimados_20_promedio": grupo["subestimados_20"].mean(),
    })


tabla = por_pliegue.groupby(["esquema", "modelo"], sort=False).apply(resumir, include_groups=False).reset_index()
tabla.to_csv("validacion_proyecto.csv", index=False)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
print(tabla.round(4).to_string(index=False))

 esquema                        modelo  mae_promedio  mae_desviacion                   mae_pliegues  costo_promedio  costo_desviacion                      costo_pliegues  subestimados_20_promedio
temporal    linea base (igual que hoy)        6.7306          0.2264   [6.98, 6.8, 6.43, 6.5, 6.95]         13.4760            0.4584  [13.86, 13.79, 12.91, 12.92, 13.9]                    0.3280
temporal gradient boosting max_depth=5        5.5787          0.2917 [5.99, 5.74, 5.31, 5.19, 5.67]         11.9107            0.9663  [12.42, 13.32, 11.27, 10.5, 12.04]                    0.3410
temporal             red neuronal (64)        5.7096          0.4890   [6.64, 5.62, 5.5, 5.2, 5.58]         12.5816            2.1138 [16.64, 12.48, 10.63, 11.53, 11.62]                    0.3359
 al azar    linea base (igual que hoy)        6.8955          0.0656 [6.89, 6.82, 7.01, 6.85, 6.91]         13.8082            0.1781 [13.83, 13.57, 14.11, 13.71, 13.82]                    0.3276
 al azar gradient bo

In [6]:
# La diferencia entre los dos modelos, pliegue por pliegue, en el esquema temporal.
temporal = por_pliegue[por_pliegue["esquema"] == "temporal"].pivot(index="pliegue", columns="modelo", values=["mae", "costo_asimetrico"])
diferencias = pd.DataFrame({
    "mae: gb - red": temporal["mae"]["gradient boosting max_depth=5"] - temporal["mae"]["red neuronal (64)"],
    "costo: gb - red": temporal["costo_asimetrico"]["gradient boosting max_depth=5"] - temporal["costo_asimetrico"]["red neuronal (64)"],
})
print(diferencias.round(4).to_string())
print("Pliegues en que el gradient boosting tiene menor MAE:", int((diferencias["mae: gb - red"] < 0).sum()), "de", len(diferencias))
print("Pliegues en que el gradient boosting tiene menor costo:", int((diferencias["costo: gb - red"] < 0).sum()), "de", len(diferencias))

         mae: gb - red  costo: gb - red
pliegue                                
1              -0.6547          -4.2219
2               0.1198           0.8421
3              -0.1925           0.6391
4              -0.0135          -1.0360
5               0.0863           0.4224
Pliegues en que el gradient boosting tiene menor MAE: 3 de 5
Pliegues en que el gradient boosting tiene menor costo: 2 de 5
